In [44]:
import os, sys, subprocess

REPO = 'projeto-itbi-cin0208'

if not os.path.exists('src/carregar_dados.py'):
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    else:
        if not os.path.isdir(REPO):
            subprocess.run(['git', 'clone', '-b', 'main',
                            f'https://github.com/CaioC11/{REPO}.git'], check=True)
        os.chdir(REPO)

sys.path.insert(0, os.getcwd())
print('Pasta atual:', os.getcwd())

Pasta atual: /content/projeto-itbi-cin0208


In [45]:
import pandas as pd
from src.carregar_dados import carregar_dados

df = carregar_dados()
categoricas = ['bairro', 'tipo_imovel', 'tipo_construcao',
               'padrao_acabamento', 'tipo_ocupacao', 'estado_conservacao']

resumo = pd.DataFrame({
    'categorias': df[categoricas].nunique(),
    'nulos': df[categoricas].isna().sum(),
    'raras (<30 linhas)': [(df[c].value_counts() < 30).sum() for c in categoricas],
})
print(resumo)

for col in ['padrao_acabamento', 'estado_conservacao', 'tipo_ocupacao']:
    print(f'\n{col}')
    print(df[col].value_counts())

                    categorias  nulos  raras (<30 linhas)
bairro                      87      0                  33
tipo_imovel                 18      0                  10
tipo_construcao             16      0                   6
padrao_acabamento            3      0                   0
tipo_ocupacao                3      0                   0
estado_conservacao           3      0                   1

padrao_acabamento
padrao_acabamento
Superior    9557
Médio       4700
Simples     2061
Name: count, dtype: int64

estado_conservacao
estado_conservacao
Bom        16118
Regular      191
Mau            9
Name: count, dtype: int64

tipo_ocupacao
tipo_ocupacao
RESIDENCIAL                    14261
COMERCIAL SEM LIXO ORGANICO     1045
COMERCIAL COM LIXO ORGANICO     1012
Name: count, dtype: int64


In [46]:
import unicodedata

def normalizar(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return ' '.join(s.lower().split())

for col in ['bairro', 'tipo_imovel', 'tipo_construcao']:
    print(col, df[col].nunique(), '->', df[col].map(normalizar).nunique())

print(df['tipo_imovel'].value_counts())
print(df['tipo_construcao'].value_counts())

bairro 87 -> 84
tipo_imovel 18 -> 18
tipo_construcao 16 -> 16
tipo_imovel
Apartamento                    13341
Casa                            1185
Sala                            1177
Loja                             221
Edificação Especial              127
Centro Comercial/Serviços         72
Galpão                            70
Hotel                             34
Instituição Financeira            23
Garagem Comercial                 21
Industria                         14
Hospital                           7
Posto de Abastecimento             6
Instituição Educacional            6
Garagem Residencial                6
Mocambo                            5
Terreno em cond residencial        2
Galpão Fechado                     1
Name: count, dtype: int64
tipo_construcao
Apartamento > 4 Pavimentos     12109
Apartamento <= 4 Pavimentos     1235
Casa                            1185
Sala > 4 Pavimentos             1127
Loja <= 4 Pavimentos             181
Edificação Especial              

In [47]:
# quais bairros estão duplicados?
agrupado = df.groupby(df['bairro'].map(normalizar))['bairro'].unique()
print(agrupado[agrupado.map(len) > 1])

# quanto tipo_construcao repete tipo_imovel?
print(df.groupby('tipo_construcao')['tipo_imovel'].nunique())

bairro
gracas                         [Gracas, Graças]
santo antonio    [Santo Antonio, Santo Antônio]
sao jose                   [Sao Jose, São José]
Name: bairro, dtype: object
tipo_construcao
Apartamento <= 4 Pavimentos    2
Apartamento > 4 Pavimentos     1
Casa                           3
Edificação Especial            2
Edificação Garagem             2
Edificação Industrial          1
Galpão                         3
Hotel                          1
Instituição Financeira         1
Instituição Hospitalar         1
Loja <= 4 Pavimentos           1
Loja > 4 Pavimentos            1
Mocambo                        1
Posto de Combustível           1
Sala <= 4 Pavimentos           1
Sala > 4 Pavimentos            2
Name: tipo_imovel, dtype: int64


In [48]:
# primeiro enconder

import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class OneHotRaras(BaseEstimator, TransformerMixin):
    """One-Hot Encoding com agrupamento opcional de categorias raras."""

    def __init__(self, colunas, min_count=None):
        self.colunas = colunas
        self.min_count = min_count

    def fit(self, X, y=None):
        self.categorias_ = {}
        self.agrupa_ = {}
        for col in self.colunas:
            contagem = X[col].value_counts()
            manter = contagem if self.min_count is None else contagem[contagem >= self.min_count]
            self.categorias_[col] = list(manter.index)
            self.agrupa_[col] = len(manter) < len(contagem)
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.colunas:
            cats = self.categorias_[col]
            if self.agrupa_[col]:
                X[col] = X[col].where(X[col].isin(cats), 'outras')
                cats = cats + ['outras']
            for cat in cats:
                X[f'{col}_{cat}'] = (X[col] == cat).astype(int)
            X = X.drop(columns=col)
        return X

    def transform(self, X):
        X = X.copy()
        novas = {}
        for col in self.colunas:
            serie = X[col]
            cats = self.categorias_[col]
            if self.agrupa_[col]:
                serie = serie.where(serie.isin(cats), 'outras')
                cats = cats + ['outras']
            for cat in cats:
                novas[f'{col}_{cat}'] = (serie == cat).astype(int)
        dummies = pd.DataFrame(novas, index=X.index)
        return pd.concat([X.drop(columns=self.colunas), dummies], axis=1)

In [49]:
cat = ['bairro', 'tipo_imovel', 'tipo_construcao',
       'padrao_acabamento', 'tipo_ocupacao', 'estado_conservacao']

novas = lambda d: [c for c in d.columns if c not in df.columns]

b0 = OneHotRaras(cat).fit(df).transform(df)
e2 = OneHotRaras(cat, min_count=30).fit(df).transform(df)
print(len(novas(b0)), len(novas(e2)))

130 84


In [50]:
class OrdinalOrdenado(BaseEstimator, TransformerMixin):
    """Ordinal encoding com a ordem definida à mão (só pra categorias com ordem natural)."""

    def __init__(self, ordens):
        self.ordens = ordens   # ex.: {'coluna': ['menor', 'meio', 'maior']}

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col, ordem in self.ordens.items():
            mapa = {cat: i for i, cat in enumerate(ordem)}
            X[col] = X[col].map(mapa)
            if X[col].isna().any():
                raise ValueError(f'{col}: há categorias fora da ordem definida {ordem}')
        return X

In [51]:
ordens = {
    'padrao_acabamento': ['Simples', 'Médio', 'Superior'],
    'estado_conservacao': ['Mau', 'Regular', 'Bom'],
}
nominais = ['bairro', 'tipo_imovel', 'tipo_construcao', 'tipo_ocupacao']

e1 = OrdinalOrdenado(ordens).fit_transform(df)
e1 = OneHotRaras(nominais).fit(e1).transform(e1)

print(e1.shape[1])
print(e1['padrao_acabamento'].value_counts())

142
padrao_acabamento
2    9557
1    4700
0    2061
Name: count, dtype: int64


In [52]:
import numpy as np

class FrequencyEncoding(BaseEstimator, TransformerMixin):
    """Troca cada categoria pela fração de linhas que ela ocupa no treino."""

    def __init__(self, colunas):
        self.colunas = colunas

    def fit(self, X, y=None):
        self.freq_ = {col: X[col].value_counts(normalize=True) for col in self.colunas}
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.colunas:
            X[col] = X[col].map(self.freq_[col]).fillna(0.0)
        return X


class TargetEncodingSuave(BaseEstimator, TransformerMixin):
    """Troca cada categoria pela média suavizada do alvo, aprendida só no treino."""

    def __init__(self, colunas, suavizacao=10):
        self.colunas = colunas
        self.suavizacao = suavizacao

    def fit(self, X, y):
        y = pd.Series(np.asarray(y), index=X.index)
        self.media_global_ = y.mean()
        self.mapas_ = {}
        for col in self.colunas:
            g = y.groupby(X[col]).agg(['mean', 'count'])
            self.mapas_[col] = ((g['count'] * g['mean'] + self.suavizacao * self.media_global_)
                                / (g['count'] + self.suavizacao))
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.colunas:
            X[col] = X[col].map(self.mapas_[col]).fillna(self.media_global_)
        return X

In [53]:
from sklearn.model_selection import train_test_split

y = np.log1p(df['valor_avaliacao'])        # alvo em log, como a Pessoa 5 vai usar
X_tr, X_te, y_tr, y_te = train_test_split(df, y, test_size=0.2, random_state=42)

alto  = ['bairro', 'tipo_imovel', 'tipo_construcao']
baixo = ['tipo_ocupacao', 'padrao_acabamento', 'estado_conservacao']

# E3: Frequency nas 3 de alta cardinalidade + One-Hot nas outras
fe = FrequencyEncoding(alto).fit(X_tr)
e3 = OneHotRaras(baixo).fit(fe.transform(X_tr)).transform(fe.transform(X_tr))

# E4: Target Encoding suavizado nas mesmas 3 + One-Hot nas outras
te = TargetEncodingSuave(alto, suavizacao=10).fit(X_tr, y_tr)
e4 = OneHotRaras(baixo).fit(te.transform(X_tr)).transform(te.transform(X_tr))

print(e3.shape[1], e4.shape[1])

# categoria que o treino nunca viu deve cair na média global
novo = X_te.head(1).copy()
novo['bairro'] = 'BAIRRO_INEXISTENTE'
print(te.transform(novo)['bairro'].iloc[0] == te.media_global_)

# os 3 bairros "mais baratos" e "mais caros" segundo o encoding (volta pra R$)
mapa = np.expm1(te.mapas_['bairro']).round(0).sort_values()
print(mapa.head(3)); print(mapa.tail(3))

28 28
True
bairro
Santo Antonio    116866.0
Recife           128466.0
Cohab   Ibura    159227.0
dtype: float64
bairro
Jaqueira     792052.0
Poco         829729.0
Monteiro    1070830.0
dtype: float64


In [54]:
class LabelArbitrario(BaseEstimator, TransformerMixin):
    """Troca cada categoria por um inteiro (ordem alfabética). Controle negativo."""

    def __init__(self, colunas):
        self.colunas = colunas

    def fit(self, X, y=None):
        self.codigos_ = {col: {cat: i for i, cat in enumerate(sorted(X[col].dropna().unique()))}
                         for col in self.colunas}
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.colunas:
            X[col] = X[col].map(self.codigos_[col]).fillna(-1).astype(int)
        return X

In [55]:
lb = LabelArbitrario(alto).fit(X_tr)
e5 = OneHotRaras(baixo).fit(lb.transform(X_tr)).transform(lb.transform(X_tr))

print(e5.shape[1])
print(list(lb.codigos_['tipo_imovel'].items())[:4])

28
[('Apartamento', 0), ('Casa', 1), ('Centro Comercial/Serviços', 2), ('Edificação Especial', 3)]
